In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.cluster import MiniBatchKMeans
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import InstanceHardnessThreshold
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.ensemble import IsolationForest
from imblearn.ensemble import BalancedBaggingClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score, average_precision_score
)

In [2]:
df = pd.read_csv(r"C:\Users\ankit\OneDrive\Desktop\Research_Papers\Online_Payments_Dataset.zip")
df.info()                     
df.describe()                 
df.nunique()                   
df.isnull().sum()              

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [3]:
df['orig_is_merchant'] = df['nameOrig'].str.startswith('M').astype(int)
df['dest_is_merchant'] = df['nameDest'].str.startswith('M').astype(int)

df = pd.get_dummies(df, columns=['type'], drop_first=True)

bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

df = df.drop(columns=['nameOrig', 'nameDest'])

print(df.shape)
print(df.select_dtypes(include='object').columns)
df.dtypes

(6362620, 14)
Index([], dtype='object')


step                  int64
amount              float64
oldbalanceOrg       float64
newbalanceOrig      float64
oldbalanceDest      float64
newbalanceDest      float64
isFraud               int64
isFlaggedFraud        int64
orig_is_merchant      int64
dest_is_merchant      int64
type_CASH_OUT         int64
type_DEBIT            int64
type_PAYMENT          int64
type_TRANSFER         int64
dtype: object

In [4]:
X = df.drop(columns=['isFraud'])
y = df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train set fraud cases:", y_train.sum(), "/", len(y_train))
print("Test set fraud cases:", y_test.sum(), "/", len(y_test), "\n")

Train set fraud cases: 6570 / 5090096
Test set fraud cases: 1643 / 1272524 



In [5]:
importance_model = RandomForestClassifier(
    n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1
)
importance_model.fit(X_train, y_train)

importances = pd.Series(
    importance_model.feature_importances_, index=X_train.columns
).sort_values(ascending=False)

print("Top 10 feature importances:")
print(importances.head(10), "\n")

threshold = 0.5 * importances.max()
selected_features = importances[importances >= threshold].index.tolist()

print("Selected features (>= 50% of top feature's importance):", selected_features, "\n")

X_train_full = X_train.copy()
X_train_full['fraud_flag'] = y_train.values

majority = X_train_full[X_train_full['fraud_flag'] == 0]
minority = X_train_full[X_train_full['fraud_flag'] == 1]

mask = pd.Series(True, index=majority.index)
for col in selected_features:
    Q1 = majority[col].quantile(0.25)
    Q3 = majority[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    mask &= majority[col].between(lower_bound, upper_bound)

majority_clean = majority[mask]

print("Majority class before IQR filtering:", len(majority))
print("Majority class after IQR filtering:", len(majority_clean), "\n")

cleaned = pd.concat([majority_clean, minority], axis=0)
X_clean = cleaned.drop(columns=['fraud_flag'])
y_clean = cleaned['fraud_flag']

print("Class balance after cleaning:", y_clean.value_counts().to_dict(), "\n")

X_train_final = cleaned.drop(columns=['fraud_flag'])
y_train_final = cleaned['fraud_flag']

print("Final class balance (No SMOTE):", pd.Series(y_train_final).value_counts().to_dict())

# sm = SMOTE(k_neighbors=3, random_state=42)
# X_train_final, y_train_final = sm.fit_resample(X_clean, y_clean)

# print("Class balance after SMOTE:", pd.Series(y_train_final).value_counts().to_dict())

Top 10 feature importances:
oldbalanceOrg       0.282777
amount              0.158165
newbalanceOrig      0.152408
type_TRANSFER       0.079627
step                0.065706
newbalanceDest      0.065199
oldbalanceDest      0.053935
type_PAYMENT        0.049880
dest_is_merchant    0.046041
type_CASH_OUT       0.045633
dtype: float64 

Selected features (>= 50% of top feature's importance): ['oldbalanceOrg', 'amount', 'newbalanceOrig'] 

Majority class before IQR filtering: 5083526
Majority class after IQR filtering: 3886493 

Class balance after cleaning: {0: 3886493, 1: 6570} 

Final class balance (No SMOTE): {0: 3886493, 1: 6570}


In [6]:
smote = SMOTE(
    sampling_strategy='auto', 
    k_neighbors=5,            
    random_state=28           
)

X_train_res, y_train_res = smote.fit_resample(X_clean, y_clean)

print("After SMOTE, training class balance:")
print(pd.Series(y_train_res).value_counts(), "\n")

After SMOTE, training class balance:
fraud_flag
0    3886493
1    3886493
Name: count, dtype: int64 



In [7]:
clf = RandomForestClassifier(
    n_estimators=150,   
    max_depth=2,         
    random_state=42,
    class_weight='balanced',
    n_jobs=-1              
)
clf.fit(X_train_res, y_train_res)

y_pred = clf.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred, digits=3))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0      1.000     0.853     0.920   1270881
           1      0.008     0.907     0.016      1643

    accuracy                          0.853   1272524
   macro avg      0.504     0.880     0.468   1272524
weighted avg      0.999     0.853     0.919   1272524

Confusion Matrix:
[[1083793  187088]
 [    152    1491]]


In [ ]:
results = []

def evaluate_model(name, clf, X_clean, y_clean, X_test, y_test, results):
    if isinstance(clf, IsolationForest):
        clf.fit(X_train)
        raw_pred = clf.predict(X_test)
        y_pred = np.where(raw_pred == -1, 1, 0)       
        scores = -clf.decision_function(X_test)            
        probs = (scores - scores.min()) / (scores.max() - scores.min())  # normalize to [0,1]
    else:
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        probs = clf.predict_proba(X_test)[:, 1] if hasattr(clf, "predict_proba") else None

    pr_auc = average_precision_score(y_test, probs) if probs is not None else np.nan

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (fraud)": precision_score(y_test, y_pred, zero_division=0),
        "Recall (fraud)": recall_score(y_test, y_pred, zero_division=0),
        "F1 (fraud)": f1_score(y_test, y_pred, zero_division=0),
        "PR-AUC": pr_auc
    })

evaluate_model("Decision Tree", 
                DecisionTreeClassifier(
                max_depth=8, 
                random_state=42),
                X_train_res, y_train_res, X_test, y_test, results)

evaluate_model("Random Forest", 
                RandomForestClassifier(
                n_estimators=300,
                max_depth=8, 
                class_weight="balanced", 
                random_state=42, 
                n_jobs=-1),
                X_train_res, y_train_res, X_test, y_test, results)

evaluate_model("Balanced Bagging", 
                BalancedBaggingClassifier(
                estimator=DecisionTreeClassifier(max_depth=2), 
                random_state=42, 
                n_jobs=-1),
                X_train, y_train, X_test, y_test, results)  

evaluate_model("XGBoost", 
                xgb.XGBClassifier(
                random_state=42, 
                max_depth=2, 
                min_child_weight=5, 
                scale_pos_weight=520,
                n_estimators=150, 
                learning_rate=0.05, 
                eval_metric="logloss"),
                X_train_res, y_train_res, X_test, y_test, results)

evaluate_model("Isolation Forest",
               IsolationForest(
                n_estimators=200,
                max_samples=256,
                contamination=0.05,
                random_state=42,
                n_jobs=-1),
                X_train, y_train, X_test, y_test, results)

evaluate_model("Balanced RF",
               BalancedRandomForestClassifier(
                n_estimators=150,
                max_depth=2,          
                sampling_strategy='all', 
                replacement=True,  
                random_state=42,
                n_jobs=-1),            
                X_train, y_train, X_test, y_test, results)

In [ ]:
# smote = SMOTE(random_state=28)
# X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# print("After SMOTE, training class balance:")
# print(y_train_res.value_counts(), "\n")

In [ ]:
# clf = xgb.XGBClassifier(
#     random_state=42, 
#     max_depth=2,          
#     min_child_weight=5,      
#     scale_pos_weight=100, 
#     n_estimators=150,        
#     learning_rate=0.05,     
#     eval_metric='logloss'
# )
# clf.fit(X_train, y_train)

# y_pred = clf.predict(X_test)

# print("Classification Report:")
# print(classification_report(y_test, y_pred, digits=3))

# print("Confusion Matrix:")
# print(confusion_matrix(y_test, y_pred))

In [ ]:
# clf = RandomForestClassifier(
#     n_estimators=150,   
#     max_depth=2,         
#     random_state=42,
#     class_weight='balanced',
#     n_jobs=-1              
# )
# clf.fit(X_train_res, y_train_res)

# y_pred = clf.predict(X_test)

# print("Classification Report:")
# print(classification_report(y_test, y_pred, digits=3))

# print("Confusion Matrix:")
# print(confusion_matrix(y_test, y_pred))

In [ ]:
# results = []

# def evaluate_model(name, clf, X_train, y_train, X_test, y_test, results):
#     if isinstance(clf, IsolationForest):
#         clf.fit(X_train)
#         raw_pred = clf.predict(X_test)
#         y_pred = np.where(raw_pred == -1, 1, 0)       
#         scores = -clf.decision_function(X_test)            
#         probs = (scores - scores.min()) / (scores.max() - scores.min())  # normalize to [0,1]
#     else:
#         clf.fit(X_train, y_train)
#         y_pred = clf.predict(X_test)
#         probs = clf.predict_proba(X_test)[:, 1] if hasattr(clf, "predict_proba") else None

#     pr_auc = average_precision_score(y_test, probs) if probs is not None else np.nan

#     results.append({
#         "Model": name,
#         "Accuracy": accuracy_score(y_test, y_pred),
#         "Precision (fraud)": precision_score(y_test, y_pred, zero_division=0),
#         "Recall (fraud)": recall_score(y_test, y_pred, zero_division=0),
#         "F1 (fraud)": f1_score(y_test, y_pred, zero_division=0),
#         "PR-AUC": pr_auc
#     })

# evaluate_model("Decision Tree", 
#                 DecisionTreeClassifier(
#                 max_depth=8, 
#                 random_state=42),
#                 X_train_res, y_train_res, X_test, y_test, results)

# evaluate_model("Random Forest", 
#                 RandomForestClassifier(
#                 n_estimators=300,
#                 max_depth=8, 
#                 class_weight="balanced", 
#                 random_state=42, 
#                 n_jobs=-1),
#                 X_train_res, y_train_res, X_test, y_test, results)

# evaluate_model("Balanced Bagging", 
#                 BalancedBaggingClassifier(
#                 estimator=DecisionTreeClassifier(max_depth=2), 
#                 random_state=42, 
#                 n_jobs=-1),
#                 X_train, y_train, X_test, y_test, results)  

# evaluate_model("XGBoost", 
#                 xgb.XGBClassifier(
#                 random_state=42, 
#                 max_depth=2, 
#                 min_child_weight=5, 
#                 scale_pos_weight=100,
#                 n_estimators=150, 
#                 learning_rate=0.05, 
#                 eval_metric="logloss"),
#                 X_train_res, y_train_res, X_test, y_test, results)

# evaluate_model("Isolation Forest",
#                IsolationForest(
#                 n_estimators=200,
#                 max_samples=256,
#                 contamination=0.05,
#                 random_state=42,
#                 n_jobs=-1),
#                 X_train, y_train, X_test, y_test, results)

# evaluate_model("Balanced RF",
#                BalancedRandomForestClassifier(
#                 n_estimators=150,
#                 max_depth=2,          
#                 sampling_strategy='all', 
#                 replacement=True,  
#                 random_state=42,
#                 n_jobs=-1),            
#                 X_train, y_train, X_test, y_test, results)

In [ ]:
# results_df = pd.DataFrame(results).sort_values("F1 (fraud)", ascending=False).reset_index(drop=True)
# print(results_df.round(4).to_string(index=False))